In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
import shutil

In [2]:
SOURCE_DIR = Path("D:/MSc/3. Spring 2026/CSE715/Project/MERGE_Lyrics_Balanced") # -> CHANGE
DEST_DIR = Path("../../../data/lyrics/en/")
METADATA_FILE = Path("../../../data/metadata/metadata_en.csv")
EMBEDDING_DIR = Path("../../../data/embeddings")

In [3]:
DEST_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
FEATURES_DIR = Path("../../../data/features/en_only")

In [5]:
all_feature_files = list(FEATURES_DIR.glob("*.npy"))
parent_ids = [f.stem.split('_')[0] for f in all_feature_files]
unique_song_ids = sorted(list(set(parent_ids)))

print(f"Total .npy files found: {len(all_feature_files)}")
print(f"Total unique songs extracted: {len(unique_song_ids)}")
print("First 10 unique IDs:", unique_song_ids[:10])

Total .npy files found: 2500
Total unique songs extracted: 1531
First 10 unique IDs: ['A002', 'A004', 'A010', 'A013', 'A014', 'A017', 'A018', 'A019', 'A029', 'A030']


In [6]:
df_meta = pd.read_csv(METADATA_FILE)
relevant_meta = df_meta[df_meta['audio_file_stem'].isin(unique_song_ids)]

In [7]:
len(relevant_meta["audio_file_stem"].unique())

1531

In [26]:
def find_lyrics_file(quadrant, stem):
    """
    Find the .txt files in "<SOURCE_DIR>/<quadrant>/<stem>.txt".
    """
    candidate = SOURCE_DIR / quadrant / f"{stem}.txt"
    return candidate if candidate.exists() else None

In [12]:
def transfer_lyrics_file(src, dst=DEST_DIR, copy_instead_of_move=True):
    """
    Creates destination folder if not already present. Copies or moves the .txt file from the source folder to the destination folder.
    """
    shutil.copy2(src, dst) if copy_instead_of_move else shutil.move(str(src), str(dst))

In [29]:
missing = []
found = 0
for _, row in relevant_meta.iterrows():
    stem = str(row["lyric_file_stem"]).strip()
    quadrant = str(row["quadrant"]).strip()
    try:
        src_file = find_lyrics_file(quadrant=quadrant, stem=stem)
        if src_file is None: 
            missing.append(stem)
            continue
        transfer_lyrics_file(src=src_file)
        found += 1
    except Exception as e:
        print(f"{e}: {stem}")

In [30]:
missing, found

([], 1531)

### Converting lyrics to word embeddings

In [ ]:
from config import config
from sentence_transformers import SentenceTransformer

In [10]:
token = config.HF_TOKEN

In [11]:
model = SentenceTransformer('all-mpnet-base-v2', token=token)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8739.96it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
for _, row in relevant_meta.iterrows():
    song_id = row['audio_file_stem']
    quadrant = row['quadrant'] # Q1, Q2, Q3, or Q4
    lyric_stem = row['lyric_file_stem']
    
    # Construct full path: LYRICS DATASET / QX / lyric_stem.txt
    lyric_path = SOURCE_DIR / quadrant / f"{lyric_stem}.txt"
    
    try:
        if lyric_path.exists():
            with open(lyric_path, 'r', encoding='utf-8') as f:
                text = f.read()
            
            # Generate vector (384 dimensions)
            z_text = model.encode(text)
            
            # Save as .npy using the song_id as name
            np.save(DEST_DIR / f"{song_id}.npy", z_text)
        else:
            print(f"Warning: File not found at {lyric_path}")
            
    except Exception as e:
        print(f"Error processing {song_id}: {e}")

In [18]:
# LYRIC_FEATURES_DIR should already be defined from our previous step
lyric_files = list(DEST_DIR.glob("*.npy"))

# 2. Print the results
print(f"Total Lyric .npy files found: {len(lyric_files)}")
print("-" * 30)

# 3. Quick check of the first few filenames to ensure they match song IDs
if len(lyric_files) > 0:
    print("Example files:")
    for f in lyric_files[:1531]:
        print(f"{f.name}")
else:
    print("Warning: No files found! Check if the path is correct.")

Total Lyric .npy files found: 1531
------------------------------
Example files:
A002.npy
A004.npy
A010.npy
A013.npy
A014.npy
A017.npy
A018.npy
A019.npy
A029.npy
A030.npy
A032.npy
A034.npy
A035.npy
A038.npy
A039.npy
A043.npy
A045.npy
A048.npy
A051.npy
A052.npy
A055.npy
A061-66.npy
A066-140.npy
A067-104.npy
A071-70.npy
A073-143.npy
A080-92.npy
A081-67.npy
A084-75.npy
A085-69.npy
A086-123.npy
A088-136.npy
A090-94.npy
A092-96.npy
A094-110.npy
A095-113.npy
A096-68.npy
A097-152.npy
A101-111.npy
A103-99.npy
A104-119.npy
A105-117.npy
A106-100.npy
A114-120.npy
A115-84.npy
A117-76.npy
A120-168.npy
A127-133.npy
A128-109.npy
A130-162.npy
A137-153.npy
A140-166.npy
A143-72.npy
A148-102.npy
A150-132.npy
A151-172.npy
A153-130.npy
A157-71.npy
A160-149.npy
A167.npy
A168.npy
A171.npy
A172.npy
A176.npy
A177.npy
A178.npy
A182.npy
A191.npy
A193.npy
A194.npy
A195.npy
A196.npy
A198.npy
MT0000008995.npy
MT0000011357.npy
MT0000026571.npy
MT0000027422.npy
MT0000031210.npy
MT0000038502.npy
MT0000040773.npy
MT000